# Exploration et préparation initiale du dataset BDD100K

L’objectif de ce notebook est de réaliser une première exploration du dataset BDD100K afin de comprendre sa structure, identifier les catégories disponibles, puis construire un premier sous-ensemble de données exploitable pour le projet.

Ce travail constitue une étape préparatoire avant l’implémentation d’un pipeline plus complet dans les scripts du dossier `data/`.

## 1. Import des bibliothèques nécessaires


In [2]:
import json
import shutil
from pathlib import Path
from collections import Counter

## 2. Chargement des annotations du jeu de données dans `bdd100k_labels_images_train.json`

L’objectif ici est de :
- charger ce fichier,
- vérifier qu’il est bien accessible,
- observer la structure générale des données.

In [3]:
json_path = Path("../data/raw/bdd100k/labels/bdd100k_labels_images_train.json")

with open(json_path, "r") as f:
    data = json.load(f)

print("Nombre total d'entrées :", len(data))
print("Clés de la première entrée :", data[0].keys())
print("Nom de la première image :", data[0]["name"])

Nombre total d'entrées : 69863
Clés de la première entrée : dict_keys(['name', 'attributes', 'timestamp', 'labels'])
Nom de la première image : 0000f77c-6257be58.jpg


## 3. Inspection d’une entrée du dataset

Avant de filtrer les données, il est nécessaire de comprendre la structure exacte d’une annotation.

Cette étape permet d’observer :
- le nom de l’image,
- les attributs globaux de la scène,
- les objets annotés présents dans l’image.

In [4]:
print(data[0])

{'name': '0000f77c-6257be58.jpg', 'attributes': {'weather': 'clear', 'scene': 'city street', 'timeofday': 'daytime'}, 'timestamp': 10000, 'labels': [{'category': 'traffic light', 'attributes': {'occluded': False, 'truncated': False, 'trafficLightColor': 'green'}, 'manualShape': True, 'manualAttributes': True, 'box2d': {'x1': 1125.902264, 'y1': 133.184488, 'x2': 1156.978645, 'y2': 210.875445}, 'id': 0}, {'category': 'traffic light', 'attributes': {'occluded': False, 'truncated': False, 'trafficLightColor': 'green'}, 'manualShape': True, 'manualAttributes': True, 'box2d': {'x1': 1156.978645, 'y1': 136.637417, 'x2': 1191.50796, 'y2': 210.875443}, 'id': 1}, {'category': 'traffic sign', 'attributes': {'occluded': False, 'truncated': False, 'trafficLightColor': 'none'}, 'manualShape': True, 'manualAttributes': True, 'box2d': {'x1': 1101.731743, 'y1': 211.122087, 'x2': 1170.79037, 'y2': 233.566141}, 'id': 2}, {'category': 'traffic sign', 'attributes': {'occluded': False, 'truncated': True, 't

## 4. Analyse des catégories d’objets présentes

Chaque image peut contenir plusieurs objets annotés.  
L’objectif de cette étape est de recenser les différentes catégories présentes dans le dataset et de mesurer leur fréquence.

Cette analyse permet ensuite de choisir les classes les plus pertinentes pour constituer un premier sous-ensemble de travail.

In [5]:
categories = Counter()

for item in data:
    for obj in item.get("labels", []):
        category = obj.get("category")
        if category:
            categories[category] += 1

print("Catégories trouvées :")
for cat, count in categories.most_common():
    print(cat, count)

Catégories trouvées :
car 713211
lane 528643
traffic sign 239686
traffic light 186117
drivable area 125723
person 91349
truck 29971
bus 11672
bike 7210
rider 4517
motor 3002
train 136


## 5. Définition d’un premier critère de filtrage

Pour cette première expérimentation, on choisit de conserver uniquement les images contenant au moins un objet de la catégorie `car`.

Ce choix permet de :
- réduire la taille du dataset,
- travailler sur une classe fréquente et pertinente pour l’analyse de scènes de conduite,
- construire un sous-ensemble simple à manipuler pour la suite du projet.

## 6. Création d’un sous-ensemble d’annotations

Dans cette étape, on parcourt l’ensemble des annotations et on conserve uniquement les images répondant au critère précédent.

On limite volontairement le nombre d’images retenues afin de :
- faciliter les premiers tests,
- éviter de manipuler un volume trop important de données,
- préparer un échantillon de travail cohérent pour le projet.

In [6]:
output_path = Path("../data/raw/bdd100k/labels/subset_car_300.json")

TARGET_CATEGORY = "car"
MAX_IMAGES = 300

source_images_candidates = [
    Path("../data/raw/bdd100k/full_images/train"),
    Path("../data/raw/bdd100k/images/100K/train"),
    Path("../data/raw/bdd100k/images/100k/train"),
    Path("../../../archive/bdd100k/bdd100k/images/100K/train"),
    Path("../../../archive/bdd100k/bdd100k/images/100k/train"),
    Path.home() / "Downloads" / "bdd100k" / "images" / "100K" / "train",
    Path.home() / "Downloads" / "bdd100k" / "images" / "100k" / "train",
    Path.home() / "Desktop" / "bdd100k" / "images" / "100K" / "train",
    Path.home() / "Desktop" / "bdd100k" / "images" / "100k" / "train",
    Path.home() / "Desktop" / "archive" / "bdd100k" / "bdd100k" / "images" / "100k" / "train",
]

source_images_dir = next((path for path in source_images_candidates if path.exists()), None)

if source_images_dir is None:
    raise FileNotFoundError(
        "Impossible de trouver le dossier d'images BDD100K train. "
        "Mets les images dans un des chemins candidats affichés dans le notebook."
    )

print("Dossier source utilisé pour le filtrage :", source_images_dir)

filtered = []
skipped_missing_image = 0

for item in data:
    labels = item.get("labels", [])
    has_target = any(obj.get("category") == TARGET_CATEGORY for obj in labels)
    image_exists = (source_images_dir / item["name"]).exists()

    if has_target and image_exists:
        filtered.append(item)
    elif has_target and not image_exists:
        skipped_missing_image += 1

    if len(filtered) >= MAX_IMAGES:
        break

with open(output_path, "w") as f:
    json.dump(filtered, f, indent=2)

print(f"Subset créé : {output_path}")
print(f"Nombre d'images gardées : {len(filtered)}")
print(f"Images ignorées car absentes localement : {skipped_missing_image}")

Subset créé : ../data/raw/bdd100k/labels/subset_car_300.json
Nombre d'images gardées : 300


## 7. Vérification du sous-ensemble généré

Après la création du fichier filtré, il est important de vérifier :
- que le sous-ensemble a bien été sauvegardé,
- qu’il contient le bon nombre d’images,
- que sa structure reste conforme à celle du fichier source.

In [23]:
with open("../data/raw/bdd100k/labels/subset_car_300.json", "r") as f:
    subset = json.load(f)

print("Nombre d'images dans le subset :", len(subset))
print("Première image :", subset[0]["name"])
print("Nombre d'objets dans la première image :", len(subset[0].get("labels", [])))

Nombre d'images dans le subset : 300
Première image : 0000f77c-6257be58.jpg
Nombre d'objets dans la première image : 11


## 8. Récupération des noms des images sélectionnées

Les annotations filtrées contiennent le nom de chaque image associée.  
Cette étape permet d’extraire cette liste afin de pouvoir copier uniquement les fichiers images correspondant au sous-ensemble retenu.

In [8]:
image_names = [item["name"] for item in subset]

print("Exemples de noms d'images :")
print(image_names[:10])

Exemples de noms d'images :
['0000f77c-6257be58.jpg', '0000f77c-cb820c98.jpg', '0001542f-5ce3cf52.jpg', '0001542f-7c670be8.jpg', '0001542f-ec815219.jpg', '0004974f-05e1c285.jpg', '00054602-3bf57337.jpg', '00067cfb-5443fe39.jpg', '00067cfb-5adfaaa7.jpg', '00067cfb-caba8a02.jpg']


## 9. Copie des images du sous-ensemble

À partir de la liste des noms extraits précédemment, on copie uniquement les images nécessaires depuis le dataset complet vers le dossier local du projet.

Cette étape permet de constituer un jeu de données réduit, cohérent avec les annotations sélectionnées, sans dupliquer l’intégralité du dataset original.

In [ ]:
if "source_images_dir" not in globals():
    source_images_candidates = [
        Path("../data/raw/bdd100k/full_images/train"),
        Path("../data/raw/bdd100k/images/100K/train"),
        Path("../data/raw/bdd100k/images/100k/train"),
        Path("../../../archive/bdd100k/bdd100k/images/100K/train"),
        Path("../../../archive/bdd100k/bdd100k/images/100k/train"),
        Path.home() / "Downloads" / "bdd100k" / "images" / "100K" / "train",
        Path.home() / "Downloads" / "bdd100k" / "images" / "100k" / "train",
        Path.home() / "Desktop" / "bdd100k" / "images" / "100K" / "train",
        Path.home() / "Desktop" / "bdd100k" / "images" / "100k" / "train",
        Path.home() / "Desktop" / "archive" / "bdd100k" / "bdd100k" / "images" / "100k" / "train",
    ]
    source_images_dir = next((path for path in source_images_candidates if path.exists()), None)

dest_images_dir = Path("../data/raw/bdd100k/images")

dest_images_dir.mkdir(parents=True, exist_ok=True)

if source_images_dir is None:
    print("Dossier source introuvable. Mets les images BDD100K train dans un des chemins suivants :")
    for candidate in source_images_candidates:
        print("-", candidate)
else:
    print("Dossier source utilisé :", source_images_dir)

copied = 0
missing = []

for name in image_names:
    if source_images_dir is None:
        missing.append(name)
        continue

    src = source_images_dir / name
    dst = dest_images_dir / name

    if src.exists():
        shutil.copy(src, dst)
        copied += 1
    else:
        missing.append(name)

print("Images copiées :", copied)
print("Images manquantes :", len(missing))

if missing:
    print("Exemples manquants :", missing[:10])

## 10. Contrôle de cohérence entre annotations et images

La dernière vérification consiste à comparer :
- le nombre d’annotations retenues,
- le nombre d’images effectivement copiées.

Cela permet de s’assurer que le sous-ensemble créé est cohérent et prêt à être utilisé dans les étapes suivantes du pipeline.

In [ ]:
raw_images_dir = Path("../data/raw/bdd100k/images")

copied_image_count = len(list(raw_images_dir.glob("*.jpg")))

print("Nombre d'annotations retenues :", len(image_names))
print("Nombre d'images présentes :", copied_image_count)
print("Nombre d'images manquantes :", len(image_names) - copied_image_count)

## 11. Conclusion

Cette première exploration a permis de :
- charger et comprendre la structure des annotations BDD100K,
- identifier les catégories présentes dans le dataset,
- construire un premier sous-ensemble ciblé sur la catégorie `car`,
- copier les images correspondantes dans l’arborescence du projet.

Ce travail constitue une base pour la suite, notamment :
- l’ajout d’autres classes,
- le filtrage par scénario (météo, moment de la journée, etc.),
- la transformation des données vers un format exploitable par le module de vision.